# 🫁 PneumoScan AI — Model Training Notebook
### Run each cell in order. Uses DenseNet-121 on your 2GB chest X-ray dataset.
---

## ✅ STEP 1 — Mount Google Drive & Upload Dataset

In [ ]:
# Mount your Google Drive (your 2GB dataset should be here)
from google.colab import drive
drive.mount('/content/drive')

# Check what's in your drive
import os
print(os.listdir('/content/drive/MyDrive/'))

# ── HOW TO UPLOAD YOUR 2GB DATASET ──────────────────────────────────────────
# Option A: Upload to Google Drive, then set path below
# Option B: Upload directly using this:
# from google.colab import files
# uploaded = files.upload()   # pick your .zip file
# !unzip chest_xray.zip -d /content/dataset/

In [ ]:
# ── SET YOUR DATASET PATH HERE ───────────────────────────────────────────────
# After upload, set this to where your data is:
DATASET_PATH = '/content/drive/MyDrive/chest_xray'  # change if needed

# Expected folder structure:
# chest_xray/
#   train/NORMAL/  train/PNEUMONIA/
#   val/NORMAL/    val/PNEUMONIA/
#   test/NORMAL/   test/PNEUMONIA/

# Verify structure
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level > 1: break

## ✅ STEP 2 — Install Libraries

In [ ]:
!pip install tensorflow==2.13.0 opencv-python matplotlib seaborn scikit-learn psycopg2-binary -q
print('✅ All libraries installed!')

## ✅ STEP 3 — Imports & GPU Check

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2, os, json, datetime
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Check GPU
print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))
print('\n✅ Go to Runtime → Change runtime type → GPU if not showing')

## ✅ STEP 4 — Data Preprocessing & Augmentation

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
IMG_SIZE   = 224        # DenseNet121 expects 224x224
BATCH_SIZE = 32
EPOCHS     = 20
NUM_CLASSES = 2         # NORMAL / PNEUMONIA (change to 4 for COVID+TB dataset)

# ── TRAINING DATA AUGMENTATION ───────────────────────────────────────────────
# Augmentation makes the model robust by showing variations of images
train_datagen = ImageDataGenerator(
    rescale=1./255,           # normalize pixel values 0-1
    rotation_range=15,        # randomly rotate images ±15 degrees
    width_shift_range=0.1,    # shift horizontally
    height_shift_range=0.1,   # shift vertically
    shear_range=0.1,          # shear transformation
    zoom_range=0.1,           # random zoom
    horizontal_flip=True,     # flip left-right (lungs are symmetric)
    fill_mode='nearest'
)

# Validation/Test — only rescale, NO augmentation
val_datagen = ImageDataGenerator(rescale=1./255)

# ── LOAD DATA ────────────────────────────────────────────────────────────────
train_gen = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb'
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False
)

test_gen = val_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False
)

# Save class names
CLASS_NAMES = list(train_gen.class_indices.keys())
print('\n📂 Classes found:', CLASS_NAMES)
print(f'Training samples:   {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Test samples:       {test_gen.samples}')

## ✅ STEP 5 — Visualize Sample Images

In [ ]:
# Peek at what the augmented data looks like
batch_imgs, batch_labels = next(train_gen)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Training Images (After Augmentation)', fontsize=14)

for i, ax in enumerate(axes.flat):
    ax.imshow(batch_imgs[i])
    label_idx = np.argmax(batch_labels[i])
    ax.set_title(CLASS_NAMES[label_idx], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150)
plt.show()
print('✅ Sample visualization complete')

## ✅ STEP 6 — Build DenseNet-121 Model (Transfer Learning)

In [ ]:
# ── HOW TRANSFER LEARNING WORKS ──────────────────────────────────────────────
# DenseNet121 was pre-trained on ImageNet (1.2M images, 1000 classes)
# We FREEZE those weights (keep the learned features)
# Then ADD our own classification head on top for chest X-ray classes
# This gives us 94%+ accuracy even with limited data!

def build_model(num_classes):
    # Load DenseNet121 WITHOUT its top classification layer
    base_model = DenseNet121(
        weights='imagenet',      # pre-trained weights
        include_top=False,       # remove ImageNet classifier
        input_shape=(224, 224, 3)
    )

    # FREEZE base model — don't update ImageNet weights during initial training
    base_model.trainable = False
    print(f'Base model: {len(base_model.layers)} layers (FROZEN)')

    # Build our classification head
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)

    x = layers.GlobalAveragePooling2D()(x)  # reduce feature maps to vector
    x = layers.BatchNormalization()(x)       # normalize activations
    x = layers.Dense(512, activation='relu')(x)  # dense layer
    x = layers.Dropout(0.4)(x)              # dropout prevents overfitting
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)  # final output

    model = models.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(NUM_CLASSES)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

model.summary()
print(f'\n✅ Model built! Total params: {model.count_params():,}')

## ✅ STEP 7 — Train Phase 1 (Frozen Base)

In [ ]:
# ── CALLBACKS ────────────────────────────────────────────────────────────────
os.makedirs('/content/saved_models', exist_ok=True)

callbacks = [
    # Save best model automatically
    ModelCheckpoint(
        '/content/saved_models/best_model.h5',
        monitor='val_auc', mode='max',
        save_best_only=True, verbose=1
    ),
    # Stop early if no improvement
    EarlyStopping(
        monitor='val_auc', patience=5,
        restore_best_weights=True, verbose=1
    ),
    # Reduce learning rate when stuck
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    )
]

# ── PHASE 1 TRAINING ─────────────────────────────────────────────────────────
print('🚀 Starting Phase 1 Training (frozen base)...')
print('This trains ONLY the new classification head\n')

history1 = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Phase 1 Training complete!')

## ✅ STEP 8 — Fine-Tuning Phase 2 (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 50 layers of DenseNet for fine-tuning
# This lets the model adapt deeper features to X-ray patterns
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen layers: {trainable_count}')

# Recompile with LOWER learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower!
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

print('\n🚀 Starting Phase 2 Fine-tuning...')
history2 = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Fine-tuning complete!')

## ✅ STEP 9 — Plot Training Curves

In [ ]:
# Combine histories
acc  = history1.history['accuracy']  + history2.history['accuracy']
vacc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
vloss= history1.history['val_loss'] + history2.history['val_loss']
auc  = history1.history['auc']  + history2.history['auc']
vauc = history1.history['val_auc'] + history2.history['val_auc']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Training History', fontsize=14)

phase_line = len(history1.history['accuracy'])

for ax, train, val, title in zip(axes,
    [acc, loss, auc], [vacc, vloss, vauc],
    ['Accuracy', 'Loss', 'AUC']):
    ax.plot(train, label=f'Train {title}', color='#00d4ff')
    ax.plot(val,   label=f'Val {title}',   color='#ff6b35')
    ax.axvline(phase_line, color='white', linestyle='--', alpha=0.4, label='Fine-tuning start')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.2)
    ax.set_facecolor('#060d14'); ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

## ✅ STEP 10 — Evaluate on Test Set

In [ ]:
print('🔍 Evaluating on test set...')
test_gen.reset()
results = model.evaluate(test_gen, verbose=1)
metrics = dict(zip(model.metrics_names, results))

print(f'\n📊 TEST RESULTS:')
print(f'  Accuracy:  {metrics["accuracy"]*100:.2f}%')
print(f'  AUC:       {metrics["auc"]*100:.2f}%')
print(f'  Precision: {metrics["precision"]*100:.2f}%')
print(f'  Recall:    {metrics["recall"]*100:.2f}%')

# Confusion Matrix
test_gen.reset()
y_pred = model.predict(test_gen, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_gen.classes

cm_matrix = confusion_matrix(y_true, y_pred_classes)
print('\nClassification Report:')
print(classification_report(y_true, y_pred_classes, target_names=CLASS_NAMES))

plt.figure(figsize=(8, 6))
sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Predicted')
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation complete')

## ✅ STEP 11 — Grad-CAM Heatmap Visualization

In [ ]:
def make_gradcam(img_array, model, last_conv_layer='conv5_block16_concat'):
    """Generate Grad-CAM heatmap showing WHERE the model is looking"""
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), predictions.numpy()

def overlay_heatmap(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255*heatmap_resized), cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(img, 1-alpha, heatmap_colored, alpha, 0)
    return cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)

# Test on a few images
test_dir = os.path.join(DATASET_PATH, 'test')
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Grad-CAM: Where the AI looks to detect disease', fontsize=13)

sample_paths = []
for cls in CLASS_NAMES[:2]:
    cls_dir = os.path.join(test_dir, cls)
    if os.path.exists(cls_dir):
        imgs = os.listdir(cls_dir)[:3]
        sample_paths += [(os.path.join(cls_dir, i), cls) for i in imgs]

for i, (path, true_label) in enumerate(sample_paths[:9]):
    ax = axes[i//3][i%3]
    img = cv2.imread(path)
    img_resized = cv2.resize(img, (224, 224))
    img_array = np.expand_dims(img_resized/255.0, axis=0)

    heatmap, preds = make_gradcam(img_array, model)
    overlay = overlay_heatmap(path, heatmap)
    pred_label = CLASS_NAMES[np.argmax(preds)]
    conf = np.max(preds)*100

    ax.imshow(overlay)
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({conf:.1f}%)', color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grad-CAM visualization complete')

## ✅ STEP 12 — Save Model + Class Names for Backend

In [ ]:
# Save the trained model
model.save('/content/saved_models/pneumoscan_model.h5')

# Save class names so backend knows what labels are
with open('/content/saved_models/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)

# Save model metrics for database storage
model_info = {
    'model_name': 'DenseNet121-PneumoScan',
    'accuracy': float(metrics['accuracy']),
    'auc': float(metrics['auc']),
    'precision': float(metrics['precision']),
    'recall': float(metrics['recall']),
    'classes': CLASS_NAMES,
    'trained_on': datetime.datetime.now().isoformat()
}
with open('/content/saved_models/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print('✅ Model saved!')
print('\n📦 Files to download:')
print('  /content/saved_models/pneumoscan_model.h5')
print('  /content/saved_models/class_names.json')
print('  /content/saved_models/model_info.json')

## ✅ STEP 13 — Download All Files to Your Computer

In [ ]:
# Download everything needed for your backend
from google.colab import files

print('Downloading model files...')
files.download('/content/saved_models/pneumoscan_model.h5')
files.download('/content/saved_models/class_names.json')
files.download('/content/saved_models/model_info.json')
files.download('/content/training_curves.png')
files.download('/content/confusion_matrix.png')
files.download('/content/gradcam_results.png')

print('\n✅ Download complete!')
print('Place pneumoscan_model.h5 in your backend/models/ folder')